# Catalogue selection methods

In this notebook, we will utilize the `Galaxy` class selection methods of galaxies stored in the `Catalogue` object to create boolean selection columns in the outputted fits table. These boolean columns can be loaded back in to new `Catalogue` objects directly from the fits catalogue upon load-in, handy for any subsequent plotting of cutouts, scaling relations, and UV LF / mass function generation. While the details on the available galaxy selection techniques are outlined in the [Galaxy selection notebook](galaxy_selection.ipynb), we will show the catalogue level implementation of the EPOCHS selection here. It is worth noting that all other galaxy selection options are also available on a catalogue scale.

In [ ]:
# imports
import astropy.units as u
from galfind.imaging.Data import morgan_version_to_dir

from galfind.catalogues.Catalogue import Catalogue
from galfind.sed_fitting.EAZY import EAZY
from galfind.selection.Selector import EPOCHS_Selector


In [ ]:
survey = "JOF"
version = "v11"
instrument_names = ["NIRCam"]
aper_diams = [0.32] * u.arcsec
forced_phot_band = ["F277W", "F356W", "F444W"]
min_flux_pc_err = 10.
SED_fit_params_arr = [
    {"templates": "fsps_larson", "lowz_zmax": 4.0},
    {"templates": "fsps_larson", "lowz_zmax": 6.0},
    {"templates": "fsps_larson", "lowz_zmax": None}
]

JOF_cat = Catalogue.pipeline(
    survey,
    version,
    instrument_names = instrument_names,
    version_to_dir_dict = morgan_version_to_dir,
    aper_diams = aper_diams,
    forced_phot_band = forced_phot_band,
    min_flux_pc_err = min_flux_pc_err
)
# load sextractor half-light radii
JOF_cat.load_sextractor_Re()

# load EAZY SED fitting results
for SED_fit_params in SED_fit_params_arr:
    EAZY_fitter = EAZY(SED_fit_params)
    EAZY_fitter(JOF_cat, aper_diams[0], load_PDFs = True, load_SEDs = True, update = True)

print(JOF_cat)

Now that we have loaded the blank catalogue, we will perform the selection. Since we are running on a `Catalogue` this time around instead of a `Galaxy` (as in [Galaxy Selection, Example 4](../selection/galaxy_selection.ipynb)), we do not need to insert the catalogue filterset when instantiating `EPOCHS_Selector` as this information is already stored in the catalogue we are running.

In [ ]:
# perform EPOCHS selection
epochs_selector = EPOCHS_Selector(aper_diams[0], EAZY_fitter)
EPOCHS_JOF_cat = epochs_selector(JOF_cat, return_copy = True)

Of course we see that returning a deep copy of the catalogue object takes longer than not. Let's have a look at how this changes the `Catalogue` print statement.

In [ ]:
print(EPOCHS_JOF_cat)

Fantastic! We now have a catalogue that has been cropped to only the EPOCHS sample.

Once the selection has been run on the `Catalogue` object, it is saved in the fits catalogue and is automatically loaded back in immediately when re-initializing the catalogue.

In [ ]:
JOF_cat_new = Catalogue.pipeline(
    survey,
    version,
    instrument_names = instrument_names,
    version_to_dir_dict = morgan_version_to_dir,
    aper_diams = aper_diams,
    forced_phot_band = forced_phot_band,
    min_flux_pc_err = min_flux_pc_err
)
# load sextractor half-light radii
JOF_cat_new.load_sextractor_Re()

# load EAZY SED fitting results
for SED_fit_params in SED_fit_params_arr:
    EAZY_fitter = EAZY(SED_fit_params)
    EAZY_fitter(JOF_cat_new, aper_diams[0], load_PDFs = True, load_SEDs = True, update = True)

In [ ]:
if JOF_cat == JOF_cat_new:
    print("Catalogues are the same")
else:
    print("Catalogues are different")

Should you notice an error in, for instance, a custom selector, you can always delete the selection fits extension so that you don't keep re-loading the dodgy results.

In [ ]:
JOF_cat_new.del_hdu(hdu = "SELECTION")

print(JOF_cat_new)